In [ ]:
import pandas as pd
import os

# Get the base directory (assuming notebook is in Code folder)
# If running from different location, adjust this path
base_dir = r'G:\Drive partagés\CS229 Project\Code'

# Read the CSV file
csv_path = os.path.join(base_dir, 'data', 'rsn_to_filename.csv')
print(f"Reading CSV from: {csv_path}")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV file not found at: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Successfully loaded {len(df)} records from CSV")

# Directory to check for files
target_dir = os.path.join(base_dir, 'Ground_motion_data', 'TimesSeries')
print(f"Checking files in: {target_dir}")

if not os.path.exists(target_dir):
    raise FileNotFoundError(f"Target directory not found: {target_dir}")

# List to store RSN values for missing files
missing_rsn = []

# Iterate through each row
for index, row in df.iterrows():
    rsn = row['Record Sequence Number']
    actual_filename = row['actual file name']

    # Construct full file path
    file_path = os.path.join(target_dir, actual_filename)

    # Check if file exists
    if not os.path.exists(file_path):
        missing_rsn.append(rsn)

# Display the list of missing RSN values
print(f"\nTotal records in CSV: {len(df)}")
print(f"Missing files: {len(missing_rsn)}")
print(f"\nList of Record Sequence Numbers with missing files:")
print(missing_rsn)


Reading CSV from: G:\Drive partagés\CS229 Project\Code\data\rsn_to_filename.csv
Successfully loaded 250 records from CSV
Checking files in: G:\Drive partagés\CS229 Project\Code\Ground_motion_data\TimesSeries

Total records in CSV: 250
Missing files: 19

List of Record Sequence Numbers with missing files:
[994, 1009, 1010, 1012, 1068, 1078, 1081, 1085, 1663, 1685, 1686, 1703, 1704, 1709, 1726, 1737, 3549, 3550, 3551]


In [ ]:
import pickle
import sys
import numpy as np

# Fix for numpy version compatibility when loading pickle files
if not hasattr(np, '_core'):
    class _CoreShim:
        numeric = np.core.numeric
    sys.modules['numpy._core'] = _CoreShim()
    sys.modules['numpy._core.numeric'] = np.core.numeric

# Create a custom unpickler that handles numpy version mismatches
class NumpyCompatibleUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith('numpy._core'):
            new_module = module.replace('numpy._core', 'numpy.core')
            try:
                return super().find_class(new_module, name)
            except:
                if 'numpy._core' not in sys.modules:
                    import numpy.core.numeric as numeric_module
                    class _CoreModule:
                        numeric = numeric_module
                    sys.modules['numpy._core'] = _CoreModule()
                    sys.modules['numpy._core.numeric'] = numeric_module
                return super().find_class(module, name)
        return super().find_class(module, name)

# Load the buildings data
pkl_path = os.path.join(base_dir, 'data', 'buildings_with_stations_v2.pkl')
print(f"Loading buildings data from: {pkl_path}")

try:
    with open(pkl_path, 'rb') as f:
        unpickler = NumpyCompatibleUnpickler(f)
        buildings_data = unpickler.load()
    print(f"Successfully loaded {len(buildings_data):,} buildings")
except Exception as e:
    print(f"Error loading pickle file: {e}")
    raise

# Convert missing_rsn to a set for faster lookup
missing_rsn_set = set(missing_rsn)
print(f"\nFiltering out {len(missing_rsn_set)} RSN values that have missing files...")

# Filter out buildings with RSN in missing_rsn
filtered_data = buildings_data[~buildings_data['closest_record_sequence_number'].isin(missing_rsn_set)]
print(f"Buildings remaining after filtering: {len(filtered_data):,}")

# Get unique RSN values (excluding missing ones)
unique_rsn = filtered_data['closest_record_sequence_number'].unique()
print(f"Unique RSN values available: {len(unique_rsn)}")

# Create subset: for each RSN, sample 20 buildings
subset_list = []
buildings_per_rsn = 20

for rsn in unique_rsn:
    rsn_buildings = filtered_data[filtered_data['closest_record_sequence_number'] == rsn]

    # Sample up to 20 buildings (or all if less than 20)
    n_samples = min(buildings_per_rsn, len(rsn_buildings))
    sampled = rsn_buildings.sample(n=n_samples, random_state=42)
    subset_list.append(sampled)

# Combine all subsets
subset_data = pd.concat(subset_list, ignore_index=True)

print(f"\nSubset created:")
print(f"  - Number of RSN values considered: {len(unique_rsn)}")
print(f"  - Total buildings in subset: {len(subset_data):,}")

# Save the subset to a new pkl file
output_path = os.path.join(base_dir, 'data', 'buildings_with_stations_subset.pkl')
print(f"\nSaving subset to: {output_path}")

with open(output_path, 'wb') as f:
    pickle.dump(subset_data, f)

print(f"Subset saved successfully!")
print(f"\nSummary:")
print(f"  - RSN values considered: {len(unique_rsn)}")
print(f"  - Buildings in subset: {len(subset_data):,}")
print(f"  - Average buildings per RSN: {len(subset_data) / len(unique_rsn):.2f}")


Loading buildings data from: G:\Drive partagés\CS229 Project\Code\data\buildings_with_stations_v2.pkl
Successfully loaded 2,233,792 buildings

Filtering out 19 RSN values that have missing files...
Buildings remaining after filtering: 2,177,064
Unique RSN values available: 140

Subset created:
  - Number of RSN values considered: 140
  - Total buildings in subset: 2,684

Saving subset to: G:\Drive partagés\CS229 Project\Code\data\buildings_with_stations_subset.pkl
Subset saved successfully!

Summary:
  - RSN values considered: 140
  - Buildings in subset: 2,684
  - Average buildings per RSN: 19.17


In [1]:
# Check file existence for unique RSN values in the subset
import pickle
import sys
import numpy as np
import pandas as pd
import os

# Use the same numpy compatibility fix
if not hasattr(np, '_core'):
    class _CoreShim:
        numeric = np.core.numeric
    sys.modules['numpy._core'] = _CoreShim()
    sys.modules['numpy._core.numeric'] = np.core.numeric

class NumpyCompatibleUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith('numpy._core'):
            new_module = module.replace('numpy._core', 'numpy.core')
            try:
                return super().find_class(new_module, name)
            except:
                if 'numpy._core' not in sys.modules:
                    import numpy.core.numeric as numeric_module
                    class _CoreModule:
                        numeric = numeric_module
                    sys.modules['numpy._core'] = _CoreModule()
                    sys.modules['numpy._core.numeric'] = numeric_module
                return super().find_class(module, name)
        return super().find_class(module, name)

# Load the subset pickle file
subset_pkl_path = r'G:\Drive partagés\CS229 Project\Code\data\buildings_with_stations_subset.pkl'
print(f"Loading subset from: {subset_pkl_path}")

try:
    with open(subset_pkl_path, 'rb') as f:
        unpickler = NumpyCompatibleUnpickler(f)
        subset_data = unpickler.load()
    print(f"Successfully loaded {len(subset_data):,} buildings from subset")
except Exception as e:
    print(f"Error loading pickle file: {e}")
    raise

# Get unique RSN values from the subset
unique_rsn = subset_data['closest_record_sequence_number'].unique()
print(f"\nFound {len(unique_rsn)} unique RSN values in subset")

# Load the CSV file to create RSN to filename mapping
csv_path = os.path.join(base_dir, 'data', 'rsn_to_filename.csv')
rsn_to_filename_df = pd.read_csv(csv_path)
print(f"Loaded {len(rsn_to_filename_df)} records from CSV mapping file")

# Create a dictionary mapping RSN to actual file name
rsn_to_filename_dict = dict(zip(
    rsn_to_filename_df['Record Sequence Number'],
    rsn_to_filename_df['actual file name']
))

# Directory to check for files
target_dir = os.path.join(base_dir, 'Ground_motion_data', 'TimesSeries')
print(f"\nChecking files in: {target_dir}")

# Check each unique RSN
results = []
missing_files = []
found_files = []

for rsn in sorted(unique_rsn):
    # Get the filename from the mapping
    if rsn in rsn_to_filename_dict:
        filename = rsn_to_filename_dict[rsn]
        file_path = os.path.join(target_dir, filename)
        file_exists = os.path.exists(file_path)

        results.append({
            'RSN': rsn,
            'Filename': filename,
            'File Exists': file_exists
        })

        if file_exists:
            found_files.append(rsn)
        else:
            missing_files.append(rsn)
    else:
        # RSN not found in mapping
        results.append({
            'RSN': rsn,
            'Filename': 'NOT FOUND IN CSV',
            'File Exists': False
        })
        missing_files.append(rsn)

# Create a results DataFrame
results_df = pd.DataFrame(results)

# Display summary
print(f"\n{'='*60}")
print(f"SUMMARY:")
print(f"{'='*60}")
print(f"Total unique RSN values: {len(unique_rsn)}")
print(f"Files found: {len(found_files)}")
print(f"Files missing: {len(missing_files)}")
print(f"\nRSN values with missing files: {missing_files}")

# Display the results table
print(f"\n{'='*60}")
print(f"DETAILED RESULTS:")
print(f"{'='*60}")
print(results_df.to_string(index=False))

# Save results to CSV if needed
results_output_path = os.path.join(base_dir, 'data', 'subset_rsn_file_check.csv')
results_df.to_csv(results_output_path, index=False)
print(f"\nResults saved to: {results_output_path}")


Loading subset from: G:\Drive partagés\CS229 Project\Code\data\buildings_with_stations_subset.pkl
Successfully loaded 2,684 buildings from subset

Found 140 unique RSN values in subset


NameError: name 'base_dir' is not defined